# Session 8: Laboratory Quality Control

**Module 3: Programming for Biological Data**  
**Date:** January 28, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Understand **Statistical Process Control** in the clinical lab
2. Create **Levey-Jennings charts** in R
3. Implement **Westgard Rules** for QC rejection
4. Detect **systematic shifts** in QC data

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL
# ============================================
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2")
library(ggplot2)

cat("✅ Setup complete!")

---

# Part 1: Statistical Process Control

## 40 minutes

---

## 1.1 QC in the Clinical Laboratory

**Quality Control (QC)** ensures that our analyzers produce accurate and precise results.

We run **control materials** with known values:
- Every day, every shift, or every batch
- Compare measured vs. expected values
- Detect problems **before** releasing patient results

## 1.2 The Levey-Jennings Chart

**Origin:** Adapted from industrial control charts (Shewhart) for clinical labs in 1950.

**Structure:**
- X-axis: Time (days, runs)
- Y-axis: QC value
- Center line: Mean
- Control limits: ±1SD, ±2SD, ±3SD

In [ ]:
# Example: Generate simulated QC data
set.seed(2024)

qc_data <- data.frame(
  Day = 1:30,
  Value = rnorm(30, mean = 100, sd = 3)
)

# Calculate control limits
mean_val <- mean(qc_data$Value)
sd_val <- sd(qc_data$Value)

cat("Mean:", round(mean_val, 2), "\n")
cat("SD:", round(sd_val, 2), "\n")
cat("\nControl Limits:\n")
cat("+3SD:", round(mean_val + 3*sd_val, 2), "\n")
cat("+2SD:", round(mean_val + 2*sd_val, 2), "\n")
cat("+1SD:", round(mean_val + 1*sd_val, 2), "\n")
cat("Mean:", round(mean_val, 2), "\n")
cat("-1SD:", round(mean_val - 1*sd_val, 2), "\n")
cat("-2SD:", round(mean_val - 2*sd_val, 2), "\n")
cat("-3SD:", round(mean_val - 3*sd_val, 2))

## 1.3 Creating a Levey-Jennings Chart

In [ ]:
# Basic Levey-Jennings chart with base R
plot(qc_data$Day, qc_data$Value,
     type = "b", pch = 19,
     main = "Levey-Jennings Chart: Glucose QC",
     xlab = "Day",
     ylab = "QC Value (mg/dL)",
     ylim = c(mean_val - 4*sd_val, mean_val + 4*sd_val))

# Add control lines
abline(h = mean_val, col = "green", lwd = 2)                    # Mean
abline(h = mean_val + 1*sd_val, col = "yellow", lty = 2)        # +1SD
abline(h = mean_val - 1*sd_val, col = "yellow", lty = 2)        # -1SD
abline(h = mean_val + 2*sd_val, col = "orange", lty = 2)        # +2SD
abline(h = mean_val - 2*sd_val, col = "orange", lty = 2)        # -2SD
abline(h = mean_val + 3*sd_val, col = "red", lwd = 2)           # +3SD
abline(h = mean_val - 3*sd_val, col = "red", lwd = 2)           # -3SD

## 1.4 Professional LJ Chart with ggplot2

In [ ]:
# Create a professional Levey-Jennings chart
ggplot(qc_data, aes(x = Day, y = Value)) +
  # Control regions (colored bands)
  geom_rect(aes(xmin = -Inf, xmax = Inf, 
                ymin = mean_val - 1*sd_val, ymax = mean_val + 1*sd_val),
            fill = "lightgreen", alpha = 0.3) +
  geom_rect(aes(xmin = -Inf, xmax = Inf,
                ymin = mean_val + 1*sd_val, ymax = mean_val + 2*sd_val),
            fill = "lightyellow", alpha = 0.3) +
  geom_rect(aes(xmin = -Inf, xmax = Inf,
                ymin = mean_val - 2*sd_val, ymax = mean_val - 1*sd_val),
            fill = "lightyellow", alpha = 0.3) +
  
  # Control lines
  geom_hline(yintercept = mean_val, color = "darkgreen", linewidth = 1) +
  geom_hline(yintercept = mean_val + 2*sd_val, color = "orange", linetype = "dashed") +
  geom_hline(yintercept = mean_val - 2*sd_val, color = "orange", linetype = "dashed") +
  geom_hline(yintercept = mean_val + 3*sd_val, color = "red", linewidth = 1) +
  geom_hline(yintercept = mean_val - 3*sd_val, color = "red", linewidth = 1) +
  
  # Data points and line
  geom_line(color = "steelblue") +
  geom_point(size = 3, color = "steelblue") +
  
  # Labels
  labs(title = "Levey-Jennings Chart: Glucose QC",
       x = "Day",
       y = "QC Value (mg/dL)") +
  theme_minimal()

## 1.5 Gaussian (Normal) Distribution Reminder

The control limits are based on the normal distribution:

| Range | % of Values | Expected Frequency |
|-------|------------|--------------------|
| ±1SD | 68.3% | ~2 in 3 |
| ±2SD | 95.4% | ~1 in 20 outside |
| ±3SD | 99.7% | ~1 in 370 outside |

**Rule of thumb:** If a value exceeds ±3SD, something is likely **wrong** with the system.

---

# Part 2: Westgard Rules

## 40 minutes

---

## 2.1 Why Westgard Rules?

Simple "outside ±3SD" isn't enough to catch all problems!

**James Westgard** developed a multi-rule system to detect:
- **Random errors** (increased imprecision)
- **Systematic errors** (bias/shift)

## 2.2 Common Westgard Rules

| Rule | Description | Error Type |
|------|-------------|------------|
| **1:3s** | 1 point > ±3SD | Random error |
| **2:2s** | 2 consecutive points > ±2SD (same side) | Systematic |
| **R:4s** | Range of 2 consecutive points > 4SD | Random error |
| **4:1s** | 4 consecutive points > ±1SD (same side) | Systematic |
| **10x** | 10 consecutive points on same side of mean | Systematic |

## 2.3 Implementing the 1:3s Rule

In [ ]:
# Add a violation to our data
qc_data$Value[15] <- mean_val + 3.5*sd_val  # Day 15 is a 1:3s violation

# Create a violation flag
qc_data$Violation_1_3s <- abs(qc_data$Value - mean_val) > 3 * sd_val

# Check violations
cat("Days with 1:3s violations:\n")
print(qc_data[qc_data$Violation_1_3s, c("Day", "Value")])

In [ ]:
# Plot with violations highlighted
ggplot(qc_data, aes(x = Day, y = Value)) +
  # Control lines
  geom_hline(yintercept = mean_val, color = "darkgreen", linewidth = 1) +
  geom_hline(yintercept = mean_val + 3*sd_val, color = "red", linewidth = 1) +
  geom_hline(yintercept = mean_val - 3*sd_val, color = "red", linewidth = 1) +
  
  # Data line
  geom_line(color = "steelblue") +
  
  # Normal points
  geom_point(data = qc_data[!qc_data$Violation_1_3s, ],
             size = 3, color = "steelblue") +
  
  # Violation points (RED!)
  geom_point(data = qc_data[qc_data$Violation_1_3s, ],
             size = 4, color = "red", shape = 17) +
  
  labs(title = "LJ Chart with 1:3s Violations Highlighted",
       subtitle = "Red triangles = violations",
       x = "Day", y = "QC Value") +
  theme_minimal()

## 2.4 Implementing the 2:2s Rule

In [ ]:
# Function to detect 2:2s violations
detect_2_2s <- function(values, mean_val, sd_val) {
  n <- length(values)
  violations <- rep(FALSE, n)
  
  for (i in 2:n) {
    # Check if both current and previous exceed 2SD on same side
    current_above_2sd <- values[i] > mean_val + 2*sd_val
    prev_above_2sd <- values[i-1] > mean_val + 2*sd_val
    
    current_below_2sd <- values[i] < mean_val - 2*sd_val
    prev_below_2sd <- values[i-1] < mean_val - 2*sd_val
    
    if ((current_above_2sd && prev_above_2sd) || 
        (current_below_2sd && prev_below_2sd)) {
      violations[i] <- TRUE
      violations[i-1] <- TRUE
    }
  }
  
  return(violations)
}

# Create test data with 2:2s violation
qc_data$Value[20] <- mean_val + 2.5*sd_val
qc_data$Value[21] <- mean_val + 2.3*sd_val  # Two consecutive > +2SD

qc_data$Violation_2_2s <- detect_2_2s(qc_data$Value, mean_val, sd_val)

cat("Days with 2:2s violations:\n")
print(qc_data[qc_data$Violation_2_2s, c("Day", "Value")])

## 2.5 Detecting Systematic Shifts

When QC values consistently move in one direction, it indicates **systematic error** (bias).

**Causes:**
- Reagent lot change
- Calibration drift
- Environmental changes

In [ ]:
# Load data with a shift
shift_data <- data.frame(
  Day = 1:60,
  Value = c(
    rnorm(30, mean = 100, sd = 2.5),  # Days 1-30: normal
    rnorm(30, mean = 106, sd = 2.5)   # Days 31-60: positive shift!
  )
)

# Calculate baseline stats from first 20 days
baseline_mean <- mean(shift_data$Value[1:20])
baseline_sd <- sd(shift_data$Value[1:20])

cat("Baseline (Days 1-20):\n")
cat("Mean:", round(baseline_mean, 2), "\n")
cat("SD:", round(baseline_sd, 2))

In [ ]:
# Visualize the shift
ggplot(shift_data, aes(x = Day, y = Value)) +
  # Control lines from baseline
  geom_hline(yintercept = baseline_mean, color = "darkgreen", linewidth = 1) +
  geom_hline(yintercept = baseline_mean + 2*baseline_sd, color = "orange", linetype = "dashed") +
  geom_hline(yintercept = baseline_mean - 2*baseline_sd, color = "orange", linetype = "dashed") +
  geom_hline(yintercept = baseline_mean + 3*baseline_sd, color = "red") +
  geom_hline(yintercept = baseline_mean - 3*baseline_sd, color = "red") +
  
  # Vertical line at shift point
  geom_vline(xintercept = 30.5, color = "purple", linetype = "dashed", linewidth = 1) +
  
  # Data
  geom_line(color = "steelblue") +
  geom_point(size = 2, color = "steelblue") +
  
  # Annotations
  annotate("text", x = 15, y = baseline_mean + 4*baseline_sd, 
           label = "Baseline", color = "darkgreen") +
  annotate("text", x = 45, y = baseline_mean + 4*baseline_sd, 
           label = "SHIFT!", color = "red", fontface = "bold") +
  
  labs(title = "Detecting a Systematic Shift",
       subtitle = "Reagent lot changed at Day 31",
       x = "Day", y = "QC Value") +
  theme_minimal()

## 2.6 QC Decision Workflow

```
Run QC sample
      ↓
Plot on LJ chart
      ↓
Check Westgard Rules
      ↓
  Violation?  →  YES  →  STOP! Investigate before releasing results
      ↓
     NO
      ↓
Release patient results
```

## 2.7 Complete QC Function

In [ ]:
# All-in-one QC assessment function
assess_qc <- function(values, baseline_mean, baseline_sd) {
  n <- length(values)
  results <- data.frame(
    Point = 1:n,
    Value = values,
    Rule_1_3s = FALSE,
    Rule_2_2s = FALSE
  )
  
  # Check 1:3s
  results$Rule_1_3s <- abs(values - baseline_mean) > 3 * baseline_sd
  
  # Check 2:2s
  for (i in 2:n) {
    above_2sd <- values[c(i-1, i)] > baseline_mean + 2*baseline_sd
    below_2sd <- values[c(i-1, i)] < baseline_mean - 2*baseline_sd
    
    if (all(above_2sd) || all(below_2sd)) {
      results$Rule_2_2s[c(i-1, i)] <- TRUE
    }
  }
  
  # Summary
  results$Any_Violation <- results$Rule_1_3s | results$Rule_2_2s
  
  return(results)
}

# Test it
qc_results <- assess_qc(qc_data$Value, mean_val, sd_val)
qc_results[qc_results$Any_Violation,]
cat("Total violations:", sum(qc_results$Any_Violation))

---

# Key Takeaways

1. **Levey-Jennings charts** visualize QC performance over time

2. **Control limits** based on Mean ± 1/2/3 SD

3. **Westgard Rules** detect different types of errors:
   - 1:3s → Random error
   - 2:2s, 4:1s, 10x → Systematic error

4. **Baseline establishment** is critical (use first 20 points)

5. **Shifts** indicate calibration drift or reagent problems

---

## Key Functions

```r
mean(values)
sd(values)
ggplot() + geom_hline() + geom_point() + geom_line()
abs(value - mean) > 3*sd  # 1:3s check
```

---

## Now proceed to Tutorial 8! 🧪